In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"

%load_ext autoreload
%autoreload 2

In [2]:
import ipdb
from Data import *
from address import *
from modelGen import *
from utils import cf_eval, cleanup_gpu
from models import CausalCondLatentCF, latentCFpp, PrototypeLatentCF
from models import CondLatentCF # Note: get the vanilla CondLatentCF
import numpy as np
import random 
from tensorflow import random as tf_random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
import argparse
import os
import pandas as pd


In [3]:
SEED = 23

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf_random.set_seed(SEED)


CF_METHODS = {  
    "LatentCFpp": latentCFpp,
    "Prototype": PrototypeLatentCF,
    "CausalCACTUS": CausalCondLatentCF,
    "CACTUS": CondLatentCF
}

In [4]:
# Experimentos que a realizar
EXP = [
    {
        "name": "LatentCF++",
        "data": "ADULT",
        "classifier": "./exp/ADULT_class/config.json",
        "AE": "./exp/ADULT_AE/config.json",
        "CFmethod": "LatentCFpp",
        "context": [
            "sex",
            "race"
        ],
        "epochs": 100,
        "tol": 0.001,
        "target_prob": 0.5,
        "learning_rate": 0.1
    },
    {
        "name": "Prototype C",
        "data": "ADULT",
        "classifier": "./exp/ADULT_class/config.json",
        "AE": "./exp/ADULT_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "sex",
            "race"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 0.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "Prototype D",
        "data": "ADULT",
        "classifier": "./exp/ADULT_class/config.json",
        "AE": "./exp/ADULT_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "sex",
            "race"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 100.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "CACTUS",
        "data": "ADULT",
        "classifier": "./exp/ADULT_class/config.json",
        "AE": "./exp/ADULT_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "sex",
            "race"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        # "dynamicAlpha": True
        "dynamicAlpha": False
    },
    {
        "name": "CausalCACTUS",
        "data": "ADULT",
        "classifier": "./exp/ADULT_class/config.json",
        "AE": "./exp/ADULT_CACTUS/config.json",
        "CFmethod": "CausalCACTUS",
        "context": [
            "sex",
            "race"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": True
        # "dynamicAlpha": False
    }
]


In [5]:
import re
from collections import defaultdict
# TODO: make a function to convert the causal paths for all the test samples
def build_causal_index_map(input_features, graph_str):
    """
    Returns:
        child_to_parents: dict {child_idx: [parent_idx, ...]}
        parent_to_children: dict {parent_idx: [child_idx, ...]}
        feat2idx: dict {feature_name: index}
    """

    # Feature name to index
    feat2idx = {f: i for i, f in enumerate(input_features)}

    # Extract edges using regex
    edges = re.findall(r'(\w+)\s*->\s*(\w+)', graph_str)

    child_to_parents = defaultdict(list)
    parent_to_children = defaultdict(list)

    for parent, child in edges:

        if parent not in feat2idx or child not in feat2idx:
            raise ValueError(f"Feature in graph not found in dataset: {parent} or {child}")

        p_idx = feat2idx[parent]
        c_idx = feat2idx[child]

        child_to_parents[c_idx].append(p_idx)
        parent_to_children[p_idx].append(c_idx)

    return dict(child_to_parents), dict(parent_to_children), feat2idx

# adult_rex5_constraints_min.dot
graph_str1 = """ 
strict digraph {
educationnum;
workclass;
occupation;
capitalloss;
sex;
race;
capitalgain;
hoursperweek;
maritalstatus;
educationnum -> occupation;
workclass -> hoursperweek;
workclass -> occupation;
capitalgain -> hoursperweek;
capitalgain -> maritalstatus;
hoursperweek -> occupation;
hoursperweek -> maritalstatus;
}
"""

# adult_rex6_constraints_mod.dot
graph_str2 = """
strict digraph {
educationnum;
workclass;
occupation;
capitalloss;
sex;
race;
capitalgain;
hoursperweek;
maritalstatus;
workclass -> hoursperweek;
workclass -> occupation;
capitalgain -> hoursperweek;
capitalgain -> occupation;
hoursperweek -> occupation;
}
"""

# # CF generation
# child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
#     data.features_lbls,
#     # graph_str
#     graph_str1
# )
# print("Feature → index:")
# print(feat2idx)
# print("\nChild → Parents (index form):")
# print(child_to_parents_dict)

In [6]:
# data.features_lbls

# # self.actionable_vars
# """
# "educationnum", "workclass", "occupation", "hoursperweek", "capitalgain", "capitalloss", "maritalstatus"
# """

## Graph 1 (Minimum priors)

In [7]:
N = 50 # TODO: rerun: unscaled LOF
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [12, 23, 34, 45, 56]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str1
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              

        # TODO: One-hot encoding for Adult
        # self.features_lbls = [
        #     "educationnum",
        #     "workclass",
        #     "occupation",
        #     "hoursperweek",
        #     "capitalgain",
        #     "capitalloss",
        #     "maritalstatus"
        # ]
        cat_idx = [1, 2, 6] #     "workclass", "occupation", "maritalstatus"
        num_idx = [0, 3, 4, 5]
        X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])
        
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict, 
            cat_idx,
            num_idx
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)




********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data
sex
1    20380
0     9782
Name: count, dtype: int64
race
1    25933
0     4229
Name: count, dtype: int64


/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Adult.py:112: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["sex"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Adult.py:113: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["race"]))






----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 15016
Positive class (>50K): 7508.0
Context 'sex=Male': 10996
Context 'race=White': 13129
----------------------------------------------------------------------------------------------------




Building model
Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 7)]               0         
                                                                 
 dense (Dense)               (None, 64)                512       
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


restoring weights of model AE: ADULT_AE
Feature → index:
{'educationnum': 0, 'workclass': 1, 'occupation': 2, 'hoursperweek': 3, 'capitalgain': 4, 'capitalloss': 5, 'maritalstatus': 6}

Child → Parents (index form):
{2: [0, 1, 3], 3: [1, 4], 6: [4, 3]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 model_2 (Functional)        (None, 7)                 4631      
                                                                 
 model (Functional)          (None, 2)                 35266     
                                                                 
Total params: 39,897
Trainable params: 39,129
Non-trainable params: 768
_________________________________________________________________
2/2 [=============================

...

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 7)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1444        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 7)            1447        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Adult.py:112: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["sex"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Adult.py:113: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["race"]))






----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 15016
Positive class (>50K): 7508.0
Context 'sex=Male': 10996
Context 'race=White': 13129
----------------------------------------------------------------------------------------------------




Building model
Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 7)]               0         
                                                                 
 dense (Dense)               (None, 64)                512       
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 7)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1444        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 7)            1447        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

In [8]:
child_to_parents_dict

{2: [0, 1, 3], 3: [1, 4], 6: [4, 3]}

In [9]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                   Validity          LOF  Compactness    Proximity  \
Dataset Model                                                              
ADULT   CACTUS        0.68 ± 0.07  1.12 ± 0.01  0.33 ± 0.03  0.65 ± 0.02   
        CausalCACTUS  0.92 ± 0.05  1.08 ± 0.01  0.47 ± 0.03  0.59 ± 0.03   
        LatentCF++    0.99 ± 0.02  1.13 ± 0.01  0.36 ± 0.03   0.6 ± 0.09   
        Prototype C     1.0 ± 0.0  2.29 ± 0.15  0.06 ± 0.01  2.83 ± 0.13   
        Prototype D     1.0 ± 0.0   2.01 ± 0.1  0.07 ± 0.01  2.57 ± 0.11   

Metric               Causal Validity (Hard) Causal Validity (Soft)  \
Dataset Model                                                        
ADULT   CACTUS                  0.95 ± 0.02            0.98 ± 0.01   
        CausalCACTUS            0.92 ± 0.06            0.97 ± 0.02   
        LatentCF++               1.0 ± 0.01              1.0 ± 0.0   
        Prototype C               1.0 ± 0.0              1.0 ± 0.0   
        Prototype D               1.0 ± 0.0              1.0 ± 0.0   

Metric               Causal Compact Validity inlier_fraction  
Dataset Model                                                 
ADULT   CACTUS                   0.34 ± 0.03       1.0 ± 0.0  
        CausalCACTUS             0.48 ± 0.03       1.0 ± 0.0  
        LatentCF++               0.33 ± 0.04     0.99 ± 0.01  
        Prototype C              0.07 ± 0.02      0.1 ± 0.09  
        Prototype D              0.08 ± 0.01     0.17 ± 0.07

In [10]:
# # self.actionable_vars
# """
# "educationnum", "workclass", "occupation", "hoursperweek", "capitalgain", "capitalloss", "maritalstatus"
# """

print(data.scaler_inverse_transform(X_test))
print(data.scaler_inverse_transform(X_cf))

[[   10     2    13    48     0     0     0]
 [   13     2    12    45     0     0     4]
 [   14     2    11    40     0     0     2]
 [   13     2    11    60     0     0     2]
 [   10     2     9    54     0     0     5]
 [   14     2     3    50     0     0     2]
 [   10     2    11    50     0     0     2]
 [   10     2     2    40     0     0     2]
 [   13     2     0    40     0     0     4]
 [   10     3    11    60     0     0     2]
 [   10     2     9    40     0     0     2]
 [   13     5     9    45     0     0     2]
 [   15     3     3    55     0  2415     2]
 [   10     2     0    40     0     0     6]
 [    9     2     0    35     0     0     4]
 [    9     2     2    40     0     0     4]
 [    9     2     7    40     0     0     4]
 [   13     2     3    40     0     0     4]
 [    6     2     5    40     0     0     0]
 [   13     2     9    50     0  1485     2]
 [   15     3     9    55     0     0     2]
 [   12     2    12    40 14344     0     4]
 [    9   

In [11]:
# self.features_lbls = [
#     "educationnum",
#     "workclass",
#     "occupation",
#     "hoursperweek",
#     "capitalgain",
#     "capitalloss",
#     "maritalstatus"
# ]
cat_idx = [1, 2, 6] #     "workclass", "occupation", "maritalstatus"
num_idx = [0, 3, 4, 5]

X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])

# --- CF evaluation ---
cf_scores, cf_scores_labels = cf_eval(
    data.X_train, # previous name: data.X_train
    context_training, # previous name: context_training,
    X_test, # previous name: x,
    X_cf, # previous name: cf_x, 
    y_original_labels, # previous name: y_,
    context_test, # previous name: context,
    y_cf_labels, # previous name: cf_y_
    data.scaler_inverse_transform(X_test),
    data.scaler_inverse_transform(X_cf),
    child_to_parents_dict,
    cat_idx,
    num_idx
)

print(cf_scores, cf_scores_labels)

[1 1 0 0 1 0 0 0 1 0 0 0 0 1 1 1 1 1 1 0 0 0 1 0 0 1 0 1 1 0 1 1 0 0 1 1 0
 0 1 0 0 1 0 1 1 1 1 0 1 1]
[1 1 0 0 1 0 0 0 1 0 0 0 1 1 1 1 1 1 1 0 0 0 1 0 0 1 1 1 1 0 1 1 0 0 1 1 0
 0 1 0 0 1 0 1 1 1 0 1 1 1]
0.92
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 586
Unique rows: 463
After np.unique(): subset size: 463
cf_samples_ shape: (2, 32)
X_ shape: (463, 32)
lof_cf min/max: 1.0954113166082757 1.2106403492019133
lof_train min/max: 0.9607420081499042 3.8237559974730537
mean train LOF: 1.0990765555438358
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 2643
Unique rows: 1685
After np.unique(): subset size: 1685
cf_samples_ shape: (12, 32)
X_ shape: (1685, 32)
lof_cf min/max: 0.9844670521418161 1.3614268424127431
lof_train min/max: 0.971333142018786 3.581064342213689
mean train LOF: 1.0679051195043303
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 942
Unique rows: 698
After np.unique(): subset size: 698
cf_samples_ shape: (1, 32)
X_ shape: (69

## Graph 2 (Moderate priors)

In [12]:
N = 50 # TODO: rerun: unscaled LOF
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [11, 22, 33, 44, 55]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str2
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              

        # TODO: One-hot encoding for Adult
        # self.features_lbls = [
        #     "educationnum",
        #     "workclass",
        #     "occupation",
        #     "hoursperweek",
        #     "capitalgain",
        #     "capitalloss",
        #     "maritalstatus"
        # ]
        cat_idx = [1, 2, 6] #     "workclass", "occupation", "maritalstatus"
        num_idx = [0, 3, 4, 5]
        X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])
        
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict, 
            cat_idx,
            num_idx
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)




********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data
sex
1    20380
0     9782
Name: count, dtype: int64
race
1    25933
0     4229
Name: count, dtype: int64




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 15016
Positive class (>50K): 7508.0
Context 'sex=Male': 10996
Context 'race=White': 13129
----------------------------------------------------------------------------------------------------




Building model


/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Adult.py:112: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["sex"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Adult.py:113: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["race"]))


Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 7)]               0         
                                                                 
 dense (Dense)               (None, 64)                512       
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                 

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Feature → index:
{'educationnum': 0, 'workclass': 1, 'occupation': 2, 'hoursperweek': 3, 'capitalgain': 4, 'capitalloss': 5, 'maritalstatus': 6}

Child → Parents (index form):
{3: [1, 4], 2: [1, 4, 3]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 model_2 (Functional)        (None, 7)                 4631      
                                                                 
 model (Functional)          (None, 2)                 35266     
                                                                 
Total params: 39,897
Trainable params: 39,129
Non-trainable params: 768
_________________________________________________________________
2/2 [==============================] - 0s 3ms/step
Generating CF 0/50
Generating CF 1

...

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 7)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1444        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 7)            1447        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Adult.py:112: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["sex"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Adult.py:113: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["race"]))


Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 7)]               0         
                                                                 
 dense (Dense)               (None, 64)                512       
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                 

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 7)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1444        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 7)            1447        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

In [13]:
child_to_parents_dict

{3: [1, 4], 2: [1, 4, 3]}

In [14]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                   Validity          LOF  Compactness    Proximity  \
Dataset Model                                                              
ADULT   CACTUS         0.7 ± 0.07  1.12 ± 0.02  0.33 ± 0.03  0.64 ± 0.05   
        CausalCACTUS   0.8 ± 0.08  1.07 ± 0.02  0.55 ± 0.03  0.47 ± 0.06   
        LatentCF++    0.99 ± 0.01  1.14 ± 0.04  0.36 ± 0.02  0.58 ± 0.09   
        Prototype C     1.0 ± 0.0  2.23 ± 0.05  0.06 ± 0.01  2.82 ± 0.09   
        Prototype D     1.0 ± 0.0  2.01 ± 0.07  0.07 ± 0.02  2.62 ± 0.08   

Metric               Causal Validity (Hard) Causal Validity (Soft)  \
Dataset Model                                                        
ADULT   CACTUS                  0.94 ± 0.04            0.97 ± 0.02   
        CausalCACTUS            0.89 ± 0.04            0.95 ± 0.02   
        LatentCF++                1.0 ± 0.0              1.0 ± 0.0   
        Prototype C               1.0 ± 0.0              1.0 ± 0.0   
        Prototype D               1.0 ± 0.0              1.0 ± 0.0   

Metric               Causal Compact Validity inlier_fraction  
Dataset Model                                                 
ADULT   CACTUS                   0.34 ± 0.03     0.97 ± 0.04  
        CausalCACTUS             0.54 ± 0.04     0.96 ± 0.02  
        LatentCF++               0.33 ± 0.03     0.97 ± 0.03  
        Prototype C              0.06 ± 0.01     0.13 ± 0.07  
        Prototype D              0.08 ± 0.02     0.24 ± 0.07